# VieNeu-TTS: Vietnamese-Native Text-to-Speech with Voice Cloning

## What is VieNeu-TTS?

**VieNeu-TTS** is a Vietnamese-native text-to-speech system with instant voice cloning capabilities. Unlike many TTS solutions that adapt English-centric architectures for Vietnamese, VieNeu-TTS was **built specifically for Vietnamese from the ground up** — it is not a fork of another model.

### Key highlights

- **Vietnamese-native architecture**: Designed from scratch for Vietnamese phonology, tones, and diacritics
- **V3 Turbo model**: 48 kHz output, bilingual (Vietnamese/English), minimal install is torch-free (runs on ONNX Runtime)
- **Voice cloning**: Clone any voice from just **3–8 seconds** of reference audio
- **Flexible deployment**: Run on **CPU** (via ONNX) or **GPU** (CUDA + PyTorch)
- **Simple installation**: `pip install vieneu` (CPU) or `pip install "vieneu[gpu]"` (GPU + legacy models)

### Resources

| Resource | Link |
|----------|------|
| GitHub | [pnnbao97/VieNeu-TTS](https://github.com/pnnbao97/VieNeu-TTS) |
| HuggingFace | [thanhtantran/VieNeu-TTS](https://huggingface.co/thanhtantran/VieNeu-TTS) |
| PyPI | [vieneu](https://pypi.org/project/vieneu/) |

## 1. Installation

VieNeu-TTS is distributed via PyPI. Choose the variant that matches your hardware:

- **GPU (recommended for Kaggle):** `pip install "vieneu[gpu]"` — includes PyTorch/CUDA dependencies and enables voice cloning + legacy models
- **CPU-only:** `pip install vieneu` — uses ONNX Runtime, no GPU required (v3 Turbo only)

In [ ]:
# Install VieNeu-TTS with GPU support (recommended on Kaggle with T4)
!pip install -q "vieneu[gpu]"

# For CPU-only environments, use instead:
# !pip install -q vieneu

## 2. Load the Model

VieNeu-TTS provides a simple Python API via the `Vieneu` class. Import it, instantiate, and you are ready to synthesize speech. Model weights are downloaded automatically on first use.

The main methods are:
- `tts.infer(text, voice=..., style=..., ref_audio=...)` — returns an audio object
- `tts.save(audio, path)` — writes the audio to a WAV file
- `tts.list_preset_voices()` — shows available built-in voices

In [ ]:
from vieneu import Vieneu

# Initialize the TTS engine (downloads model weights on first run)
# Defaults to v3 Turbo (48 kHz); auto-detects CPU/GPU
tts = Vieneu()
print("VieNeu-TTS model loaded successfully.")

# List available built-in voices
voices = tts.list_preset_voices()
print("\nAvailable preset voices:")
for label, voice_id in voices:
    print(f"  - {label} ({voice_id})")

## 3. Basic Synthesis

Let's start with a simple example — synthesize a single Vietnamese sentence and save it to a WAV file.

In [ ]:
# Synthesize a simple greeting
audio = tts.infer("Xin chào, tôi là VieNeu.")
tts.save(audio, "output_hello.wav")
print("Saved: output_hello.wav")

In [ ]:
# Play the audio in the notebook
from IPython.display import Audio, display

display(Audio("output_hello.wav"))

## 4. Testing with Multiple Vietnamese Texts

Vietnamese is a tonal language with six tones. A good TTS system must handle tones, diacritics, and natural prosody correctly. Let's test with a variety of sentences and reading styles.

Available styles: `"tu_nhien"` (natural/conversational), `"tin_tuc"` (news), `"doc_truyen"` (storytelling).

In [ ]:
test_sentences = [
    ("Hôm nay thời tiết rất đẹp.", "tu_nhien", "output_weather.wav"),
    ("Việt Nam là một đất nước xinh đẹp với nhiều danh lam thắng cảnh.", "doc_truyen", "output_vietnam.wav"),
    ("Trí tuệ nhân tạo đang thay đổi thế giới mỗi ngày.", "tin_tuc", "output_ai.wav"),
    ("Cảm ơn bạn đã sử dụng VieNeu Text-to-Speech.", "tu_nhien", "output_thanks.wav"),
]

for text, style, path in test_sentences:
    audio = tts.infer(text, style=style)
    tts.save(audio, path)
    print(f"Synthesized ({style}): {text}")
    print(f"  -> {path}")

print("\nAll test sentences synthesized.")

In [ ]:
# Listen to each result
from IPython.display import Audio, display, Markdown

for text, style, path in test_sentences:
    display(Markdown(f"**{text}** *(style: {style})*"))
    display(Audio(path))

## 5. Voice Cloning

One of VieNeu-TTS's standout features is **instant voice cloning**. You provide a short reference audio clip (3–8 seconds) of a target speaker, and the model synthesizes new text in that voice.

**Note:** Voice cloning requires the GPU backend (`pip install "vieneu[gpu]"`).

### How it works
1. Record or upload a short audio clip of the target speaker (WAV format, 3–8 seconds)
2. Pass `ref_audio="path/to/clip.wav"` to `tts.infer()`
3. The model extracts speaker characteristics and generates speech in the cloned voice
4. Optionally use `denoise=True` to clean background noise from the reference

In [ ]:
# For demonstration, we use one of our earlier outputs as the reference voice.
# In practice, you would use a real recording of your target speaker.
reference_audio = "output_hello.wav"

# Synthesize new text using the cloned voice
audio = tts.infer(
    "Chào mừng bạn đến với công nghệ giọng nói nhân tạo.",
    ref_audio=reference_audio,
    denoise=True,
)
tts.save(audio, "output_cloned.wav")
print("Voice cloning synthesis complete.")

# You can also save and reuse a cloned voice by name:
# tts.add_voice("My Voice", "reference.wav")
# audio = tts.infer("Text here", voice="My Voice")

In [ ]:
# Compare original voice vs cloned voice
from IPython.display import Audio, display, Markdown

display(Markdown("**Reference audio (original speaker):**"))
display(Audio(reference_audio))

display(Markdown("**Cloned voice (new text, same speaker characteristics):**"))
display(Audio("output_cloned.wav"))

## 6. Key Takeaways

| Feature | Detail |
|---------|--------|
| **Vietnamese-native** | Built from the ground up for Vietnamese — not a fork of an English TTS model |
| **V3 Turbo** | 48 kHz output, minimal install is torch-free (ONNX Runtime on CPU) |
| **Easy installation** | `pip install vieneu` (CPU) or `pip install "vieneu[gpu]"` (GPU + voice cloning) |
| **Reading styles** | `tu_nhien` (natural), `tin_tuc` (news), `doc_truyen` (storytelling) |
| **Voice cloning** | Clone any voice from 3–8 seconds of reference audio via `ref_audio=` |
| **Emotion cues** | Experimental inline tags: `[cười]`, `[thở dài]`, `[hắng giọng]` |
| **Simple API** | `tts.infer()` to generate, `tts.save()` to write WAV |

### Next steps

- Explore the [GitHub repo](https://github.com/pnnbao97/VieNeu-TTS) for advanced configuration options
- Try voice cloning with your own recordings
- Check the [HuggingFace model page](https://huggingface.co/thanhtantran/VieNeu-TTS) for model cards and benchmarks
- Experiment with longer texts, mixed Vietnamese/English input, and different speakers